# LangChain AI Agent Tutorial (OpenRouter)

This notebook demonstrates building AI applications with **LangChain**, using the **OpenRouter** API key.

It uses LangChain abstractions instead of raw HTTP requests:

1. **Part 1 — LangChain Basics**: Prompts, models, and LCEL chains (based on the [GeeksforGeeks LangChain intro](https://www.geeksforgeeks.org/artificial-intelligence/introduction-to-langchain/))
2. **Part 2 — LangChain Agent with Tools**: Function calling via `@tool` and `create_agent`
3. **Part 3 — LangChain RAG**: Document retrieval and context-aware answers

---

## What is LangChain?

LangChain is an open-source framework for building LLM-powered applications. Key components:

<div>
<img src="images/Langchain1.png" width="1000"/>
</div>

| Component | Purpose |
|-----------|--------|
| **Chains** | Sequence of steps (prompt → model → parser) composed with LCEL (`\|`) |
| **Prompt Management** | Reusable templates with placeholders like `{year}` |
| **Agents** | LLM-driven components that choose and call tools |
| **Vector Database** | Stores embeddings for similarity search (RAG) |
| **Models** | Unified interface to many LLM providers |
| **Memory** | Keeps conversation context across turns |

**Prerequisite:** Copy `env.example` to `.env` and set `OPENROUTER_API_KEY`.

## Step 1: Install Dependencies

Install LangChain core packages and the official OpenRouter integration.

- `langchain` / `langchain-core`: Chains, prompts, parsers, tools
- `langchain-openrouter`: Native OpenRouter chat model (`ChatOpenRouter`)
- `langchain-community` / `langchain-text-splitters`: Document loaders and text splitting (RAG)
- `langchain-openai`: OpenAI-compatible embeddings via OpenRouter
- `faiss-cpu` / `pypdf`: Vector store and PDF loading for Part 3

In [ ]:
#!pip install -q langchain langchain-core langchain-openrouter langchain-community langchain-text-splitters langchain-openai faiss-cpu pypdf unstructured

---

## Part 1: LangChain Basics

This section follows the GeeksforGeeks tutorial structure, adapted for OpenRouter.

### Steps covered:
1. Import libraries
2. Configure API key (from `.env`)
3. Initialize the chat model
4. Run a simple prompt
5. Build a **Prompt Template** and **LCEL chain**

### Step 2–4: Import Libraries, Load API Key, Initialize Model

The GeeksforGeeks article uses `ChatGoogleGenerativeAI`. Here we use **`ChatOpenRouter`**, which reads `OPENROUTER_API_KEY` from the environment automatically.

- **`temperature`**: Controls creativity (0 = deterministic, 1 = more creative)
- **`model`**: Any model available on [OpenRouter](https://openrouter.ai/models) (e.g. `google/gemini-2.5-flash`)

In [1]:
import os
import textwrap
from dotenv import load_dotenv

from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load OPENROUTER_API_KEY from .env
load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("Set OPENROUTER_API_KEY in your .env file before running this notebook.")

# Same default model as AI_Agent_test.ipynb
MODEL = "google/gemini-2.5-flash"

# Free models you can try on OpenRouter:
# MODEL = "openai/gpt-oss-20b:free"
# MODEL = "google/gemma-3-27b-it:free"
# MODEL = "qwen/qwen3-coder:free"

# Initialize the LangChain chat model via OpenRouter
llm = ChatOpenRouter(
    model=MODEL,
    temperature=0.7,
)

print(f"Model ready: {MODEL}")

Model ready: google/gemini-2.5-flash


### Step 5: Run a Simple Prompt

Send a plain-text prompt directly to the model with `.invoke()`. This is the simplest way to interact with an LLM — no templates or chains yet.

In [3]:
prompt = "Suggest me a list of AI skills that is in demand?"

# .invoke() sends the prompt to the LLM and returns an AIMessage
response = llm.invoke(prompt)

# .content holds the text reply
print("Suggested Skill:\n", textwrap.fill(response.content, width=100))

Suggested Skill:
 The field of AI is evolving rapidly, so staying current is key. Here's a comprehensive list of AI
skills that are currently in high demand, categorized for clarity:  ---  **I. Core Technical Skills
(Foundational & Essential)**  1.  **Programming Languages:**     *   **Python:** Absolutely dominant
in AI/ML due to its rich libraries (NumPy, SciPy, Pandas, Scikit-learn, TensorFlow, PyTorch).     *
**R:** Strong in statistical analysis and data visualization, particularly in academia and
specialized data science roles.     *   **Java/C++:** Important for high-performance computing,
large-scale systems, and deploying AI models in production environments.  2.  **Mathematics &
Statistics:**     *   **Linear Algebra:** Essential for understanding algorithms (e.g., PCA, matrix
operations in neural networks).     *   **Calculus (Differential & Integral):** Crucial for
optimization algorithms (gradient descent) and understanding model training.     *   **Probability &
Statistic

### Steps 6–9: Prompt Template, Parser, and LCEL Chain

**LCEL (LangChain Expression Language)** composes workflows with the `|` (pipe) operator:

```
prompt_template  →  llm  →  StrOutputParser()
     ↓                ↓            ↓
  Fill {year}    Call model    Return string
```

1. **`PromptTemplate`**: Replaces placeholders (e.g. `{year}`) with input values
2. **`llm`**: Sends the formatted prompt to OpenRouter
3. **`StrOutputParser()`**: Extracts clean string text from the model response

In [4]:
# Step 6: Create a dynamic prompt template with a {year} placeholder
template = "Give me 3 career skills that are in high demand in {year}."
prompt_template = PromptTemplate.from_template(template)

# Step 7: Parser ensures the output is returned as a plain string
parser = StrOutputParser()

# Step 8: Build the LCEL chain — data flows left to right through each step
chain = prompt_template | llm | parser

# Step 9: Run the chain; {year} is replaced with "2026"
response = chain.invoke({"year": "2026"})
print("\nCareer Skills in 2026:\n", textwrap.fill(response, width=100))


Career Skills in 2026:
 Predicting specific in-demand skills for 2026 with absolute certainty is tough, as the job market is
dynamic. However, based on current trends and the trajectory of technological and societal
development, here are three career skills likely to be in high demand:  1.  **AI Literacy and Prompt
Engineering:** This goes beyond just understanding what AI is. It encompasses:     *
**Understanding AI Capabilities and Limitations:** Knowing what various AI models (generative AI,
predictive AI, etc.) can realistically achieve, what their biases might be, and where human
oversight is crucial.     *   **Prompt Engineering:** The ability to craft precise, effective, and
nuanced prompts to get the best results from AI tools (like ChatGPT, Midjourney, etc.). This
involves understanding how to guide AI, refine outputs, troubleshoot issues, and integrate AI into
workflows for increased productivity and innovation.     *   **Ethical AI Application:** Recognizing
the ethical imp

---

## Part 2: LangChain Agent with Tools

In `AI_Agent_test.ipynb`, tools are defined as JSON schemas and passed to the OpenRouter API manually. LangChain simplifies this:

1. **`@tool` decorator**: Wraps Python functions; LangChain auto-generates the schema
2. **`create_agent()`**: Runs the tool-calling loop (decide → call → respond)
3. **Same tools** as the native demo: `calculate`, `get_current_time`, `text_uppercase`, `text_word_count`

### How it works:
```
User query → Agent (LLM) → Tool call? → Execute function → Final answer
```

In [5]:
import math
from datetime import datetime

from langchain_core.tools import tool
from langchain.agents import create_agent


@tool
def calculate(expression: str) -> str:
    """Perform mathematical calculations. Supports +, -, *, / and math functions like sqrt, sin, cos."""
    try:
        allowed_names = {
            k: v for k, v in math.__dict__.items() if not k.startswith("__")
        }
        allowed_names.update({"abs": abs, "round": round, "min": min, "max": max})
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def text_uppercase(text: str) -> str:
    """Convert text to uppercase letters."""
    return text.upper()


@tool
def text_word_count(text: str) -> str:
    """Count the number of words in a given text."""
    return f"Word count: {len(text.split())}"


# Collect all tools for the agent
tools = [calculate, get_current_time, text_uppercase, text_word_count]

# Create the agent: LangChain handles the tool-calling loop automatically
agent = create_agent(
    model=llm,
    tools=tools,
)


def chat_with_agent(user_message: str) -> str:
    """Send a message to the agent and return the final text response."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_message}]}
    )

    # Print tool calls for transparency (similar to AI_Agent_test.ipynb)
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print("Tool calls:")
            for tc in msg.tool_calls:
                print(f"  -> {tc['name']}({tc['args']})")

    # Last message is the assistant's final answer
    final = result["messages"][-1]
    return final.content if hasattr(final, "content") else str(final)


print("Agent ready with tools:", [t.name for t in tools])

Agent ready with tools: ['calculate', 'get_current_time', 'text_uppercase', 'text_word_count']


In [7]:
print("=" * 80)
print("LangChain Agent — Tool Calling Demo")
print("=" * 80)

# Example 1: Mathematical calculation
print("\nExample 1: Mathematical Calculation")
print("-" * 80)
query1 = "What is the square root of 144 plus 25?"
answer1 = chat_with_agent(query1)
print(f"\nUser: {query1}")
print(f"\nAI Response:\n{textwrap.fill(answer1, width=80)}")

# Example 2: Get current time
print("\n\nExample 2: Get Current Time")
print("-" * 80)
query2 = "What time is it now?"
answer2 = chat_with_agent(query2)
print(f"\nUser: {query2}")
print(f"\nAI Response:\n{textwrap.fill(answer2, width=80)}")

# Example 3: Text processing
print("\nExample 3: Text Processing")
print("-" * 80)
query3 = "Convert 'Hello World' to uppercase and count the words"
answer3 = chat_with_agent(query3)
print(f"\nUser: {query3}")
print(f"\nAI Response:\n{textwrap.fill(answer3, width=80)}")

LangChain Agent — Tool Calling Demo

Example 1: Mathematical Calculation
--------------------------------------------------------------------------------
Tool calls:
  -> calculate({'expression': 'sqrt(144) + 25'})

User: What is the square root of 144 plus 25?

AI Response:
The square root of 144 plus 25 is 37.


Example 2: Get Current Time
--------------------------------------------------------------------------------
Tool calls:
  -> get_current_time({})

User: What time is it now?

AI Response:
The current time is 14:43:05 on July 31, 2026.

Example 3: Text Processing
--------------------------------------------------------------------------------
Tool calls:
  -> text_uppercase({'text': 'Hello World'})
  -> text_word_count({'text': 'Hello World'})

User: Convert 'Hello World' to uppercase and count the words

AI Response:
I have converted 'Hello World' to uppercase: 'HELLO WORLD'. Also, the word count
for 'Hello World' is 2.


### Try Your Own Queries

Modify the query below. The agent will automatically choose which tools to call.

In [8]:
your_query = "Calculate 15 * 8 and tell me what time it is"  # Modify this query

answer = chat_with_agent(your_query)
print(f"\nUser: {your_query}")
print(f"\nAI Response:\n{textwrap.fill(answer, width=80)}")

Tool calls:
  -> calculate({'expression': '15 * 8'})
  -> get_current_time({})

User: Calculate 15 * 8 and tell me what time it is

AI Response:
The result of 15 * 8 is 120. The current time is 14:43:14 on July 31, 2026.


---

## Part 3: LangChain RAG (Retrieval-Augmented Generation)

**RAG** enhances LLM answers by retrieving relevant document chunks before generating a response.

### RAG Workflow:
```
PDF docs → Split into chunks → Create embeddings → Store in FAISS
                                                              ↓
User question → Embed query → Similarity search → Top-k chunks → LLM answer
```

### Key components:
- **`PyPDFLoader`**: Loads PDF files from the `docs/` folder
- **`RecursiveCharacterTextSplitter`**: Splits text into chunks (500 tokens, 50 overlap — same as `VectorStore_v2.py`)
- **`OpenAIEmbeddings`**: Creates vectors via OpenRouter (`openai/text-embedding-3-small`)
- **`FAISS`**: In-memory vector store for fast similarity search
- **RAG LCEL chain**: Retrieves context, then generates an answer

### Step 1: Build the Vector Store from Documents

Load PDFs from `docs/`, split them, embed each chunk, and save to `vectorstore/langchain_faiss/`.

On subsequent runs, you can skip this cell and load the saved index instead (see the optional load cell below).

In [12]:
import glob
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings as OpenRouterEmbeddings
from langchain_community.vectorstores import FAISS

DOCS_DIR = "docs"
FAISS_PATH = "vectorstore/langchain_faiss"
EMBEDDING_MODEL = "openai/text-embedding-3-small"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# OpenRouter is OpenAI-compatible for embeddings, so we reuse OpenAIEmbeddings
# and point it to OpenRouter's endpoint.
embeddings = OpenRouterEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

# Load supported document files from the docs folder
supported_patterns = ["*.pdf", "*.docx", "*.txt"]
files_to_process = []
for pattern in supported_patterns:
    files_to_process.extend(glob.glob(os.path.join(DOCS_DIR, pattern)))

files_to_process = sorted(set(files_to_process))

if not files_to_process:
    raise FileNotFoundError(
        f"No supported files found in '{DOCS_DIR}/'. Add a PDF, DOCX, or TXT file before running Part 3."
    )

documents = []
for file_path in files_to_process:
    print(f"Loading: {file_path}")
    if file_path.lower().endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file_path.lower().endswith(".docx"):
        loader = UnstructuredWordDocumentLoader(file_path)
    else:
        loader = TextLoader(file_path, encoding="utf-8")

    documents.extend(loader.load())

print(f"Loaded {len(documents)} page/chunk(s) from {len(files_to_process)} file(s)")

# Split documents into smaller chunks for retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} text chunks")

# Build FAISS index and persist to disk
vectorstore = FAISS.from_documents(chunks, embeddings)
os.makedirs(os.path.dirname(FAISS_PATH), exist_ok=True)
vectorstore.save_local(FAISS_PATH)
print(f"Vector store saved to: {FAISS_PATH}")

Loading: docs/DjamTechnologies presention_Francais.docx
Loaded 1 page/chunk(s) from 1 file(s)
Created 11 text chunks
Vector store saved to: vectorstore/langchain_faiss


In [ ]:
# Optional: Load a previously saved FAISS index (skip the build cell above)
# vectorstore = FAISS.load_local(FAISS_PATH, embeddings, allow_dangerous_deserialization=True)
# print(f"Loaded vector store from: {FAISS_PATH}")

### Step 2: Query the Knowledge Base


<div>
<img src="images/Langchain2.png" width="1000"/>
</div>

Build a RAG chain that:
1. Retrieves the top-k most similar chunks for the user's question
2. Injects them into a prompt as context
3. Sends the prompt to the LLM for a grounded answer

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Retriever: returns top 5 most similar chunks (same k as querykb_v2.py default)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# RAG prompt: instructs the LLM to answer only from the provided context
rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}"""
)


def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


# LCEL RAG chain: retrieve context + question → prompt → LLM → string
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)


def ask_rag(question: str) -> tuple[str, str]:
    """Ask a question and return (answer, context_used)."""
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    answer = rag_chain.invoke(question)
    return answer, context


# Sample query (same topic as AI_Agent_test.ipynb)
user_msg = (
    "What is Djam Technologies? "
    "Who is the CEO of Djam Technologies, and what are the other team members? "
)

answer, used_context = ask_rag(user_msg)

print(f"User Question: {user_msg}")
print(f"\nAI Response:\n{textwrap.fill(answer, width=80)}")
# Uncomment to inspect retrieved chunks:
# print("\nContext used:\n", used_context[:500], "...")

User Question: What is Djam Technologies? Who is the CEO of Djam Technologies, and what are the other team members? 

AI Response:
DJAM Technologies is a Cameroonian startup created in 2018 that develops and
manufactures electronic equipment, artificial intelligence applications, and
software in the fields of medicine, security, agriculture, and livestock.  The
CEO of DJAM Technologies is Youssoufa Mohamadou.  The other team members are: *
Njike Kouekeu Landry (Chief Technical Officer - CTO) * Kuum Cletus (Chief
Software/AI Developer) * Djemi Yvan (Chief Hardware Developer) * David Kegne
(Hardware Developer / Logistic Officer) * Khadidja Aminou (Software Developer) *
Kel Momo (Software Developer) * Koneh Dulas (Hardware Developer) * Dongmo L.
Dirane (Hardware Developer)


### Try Your Own RAG Questions

Modify the query below to ask questions about your documents.

In [15]:
your_rag_query = "What is this document about?"  # Modify this query

answer, _ = ask_rag(your_rag_query)
print(f"User Question: {your_rag_query}")
print(f"\nAI Response:\n{textwrap.fill(answer, width=80)}")

User Question: What is this document about?

AI Response:
This document is about DJAM Technologies, a Cameroonian startup founded in 2018.
It details the company's team, activities, and specific branches like DJAM
Security Tech. and Djam AI Agents (DAA). The company focuses on developing and
manufacturing electronic equipment, AI applications, and software in various
domains like medicine, security, agriculture, and livestock, aiming to solve
local problems using new technologies.


---

## Summary

### What We've Demonstrated

1. **LangChain Basics (Part 1)**: Prompt templates and LCEL chains with OpenRouter
2. **LangChain Agent (Part 2)**: Tool calling with `@tool` and `create_agent`
3. **LangChain RAG (Part 3)**: Document loading, FAISS vector store, and retrieval-augmented answers

### Native API vs LangChain

| Feature | Native API (`AI_Agent_test.ipynb`) | LangChain (`Ai_Agent_with_Langchain.ipynb`) |
|---------|--------------------------------------|---------------------------------------------|
| LLM calls | Manual `requests.post` to OpenRouter | `ChatOpenRouter.invoke()` |
| Chains | Not applicable | LCEL pipe: `prompt \| llm \| parser` |
| Tool schemas | Hand-written JSON schemas | Auto-generated from `@tool` functions |
| Tool loop | Manual two-step API calls | `create_agent()` handles the loop |
| Vector store | Custom JSON + numpy cosine similarity | FAISS via `langchain-community` |
| RAG pipeline | Custom `RAG` class in `querykb_v2.py` | LCEL chain with retriever |
| Dependencies | Minimal (`requests`, `openai`, `numpy`) | LangChain ecosystem packages |

### When to use which?

- **Native API**: Lightweight, full control, fewer dependencies — good for learning HTTP/API fundamentals
- **LangChain**: Faster development, reusable abstractions, rich ecosystem — good for production apps and complex workflows

### Next Steps

- Try different OpenRouter models in Part 1 and Part 2
- Add more PDFs to `docs/` and rebuild the FAISS index
- Experiment with `chunk_size`, `chunk_overlap`, and retriever `k` in Part 3
- Optional: enable [LangSmith tracing](https://docs.smith.langchain.com/) by setting `LANGSMITH_API_KEY` and `LANGSMITH_TRACING=true`